# 02 — Train & test

Build a dataset, train the `causal` model (multi-seed), and run the test-gate metrics (per-output regression, calibration, proportion-common-cause vs disparity).

**Note:** requires the `TODO(science)` analytical observer so that valid targets exist.

In [ ]:
from pathlib import Path
import numpy as np

from causal_msi.config import load_config
from causal_msi.generative import build_dataset
from causal_msi.training import run_multiseed
from causal_msi.utils import seed_everything

cfg = load_config(Path('..') / 'configs' / 'default.yaml')
rng = seed_everything(cfg.seed)
# Use a smaller dataset for a quick notebook run.
dataset = build_dataset(rng, cfg, n_trials=5000)
dataset.X.shape, dataset.Y.shape

In [ ]:
results = run_multiseed(cfg, checkpoint_dir='../checkpoints', dataset=dataset)
for r in results:
    print(f'seed {r.seed}: best_val={r.best_val:.4f} @ epoch {r.best_epoch}')

In [ ]:
# Test-gate: per-output regression decoded-vs-analytical.
import torch
from causal_msi.analysis.performance import per_output_regression

model = results[0].model.eval()
with torch.no_grad():
    pred = model(torch.as_tensor(dataset.X, dtype=torch.float32)).cpu().numpy()
for reg in per_output_regression(pred, dataset.Y):
    print(reg)